# graphio — Demo

**graphio** is a Python library for bulk-loading data into Neo4j.

https://github.com/kaiserpreusse/graphio

Two main APIs:
1. low-level bulk loading
2. Pydantic based OGM

> **Before running:** `docker compose up -d` — Neo4j community at `bolt://localhost:13687` (auth: `neo4j/test`).

In [1]:
from neo4j import GraphDatabase
from graphio import NodeSet, RelationshipSet, NodeModel, Relationship, Base, CypherQuery, RelField

driver = GraphDatabase.driver("bolt://localhost:13687", auth=("neo4j", "test"))
driver.verify_connectivity()
print("Connected ✓")

Connected ✓


---
## 1 — NodeSet: Bulk-loading nodes

A `NodeSet` holds nodes that share the same **labels** and **merge keys**. Merge keys are the properties that define node uniqueness — used for both MERGE operations and index creation.

In [2]:
people = NodeSet(
    labels=["Person"],
    merge_keys=["email"]
)

people.add({"email": "alice@example.com", "name": "Alice", "age": 30})
people.add({"email": "bob@example.com",   "name": "Bob",   "age": 25})
people.add({"email": "carol@example.com", "name": "Carol", "age": 35})
people.add({"email": "dave@example.com",  "name": "Dave",  "age": 28})
people.add({"email": "eve@example.com",   "name": "Eve",   "age": 32})

print(f"{people}")
print(f"Nodes in memory: {len(people.nodes)}")

<NodeSet (['Person']; ['email'])>
Nodes in memory: 5


In [3]:
# Always create the index before merging — it makes MERGE fast
people.create_index(driver)
people.merge(driver)
print("Done.")

Done.


In [4]:
# Helper to run ad-hoc Cypher queries
def query(cypher, **params):
    with driver.session() as session:
        rows = session.run(cypher, **params).data()
    for row in rows:
        print(row)
    return rows

query("MATCH (p:Person) RETURN p.name AS name, p.email AS email, p.age AS age ORDER BY p.name")

{'name': 'Alice', 'email': 'alice@example.com', 'age': 30}
{'name': 'Bob', 'email': 'bob@example.com', 'age': 25}
{'name': 'Carol', 'email': 'carol@example.com', 'age': 35}
{'name': 'Dave', 'email': 'dave@example.com', 'age': 28}
{'name': 'Eve', 'email': 'eve@example.com', 'age': 32}


[{'name': 'Alice', 'email': 'alice@example.com', 'age': 30},
 {'name': 'Bob', 'email': 'bob@example.com', 'age': 25},
 {'name': 'Carol', 'email': 'carol@example.com', 'age': 35},
 {'name': 'Dave', 'email': 'dave@example.com', 'age': 28},
 {'name': 'Eve', 'email': 'eve@example.com', 'age': 32}]

### CREATE vs MERGE

| Operation | What it does | When to use |
|-----------|-------------|-------------|
| `.create(driver)` | `CREATE` — fastest, no duplicate check | Fresh load, data is guaranteed unique |
| `.merge(driver)` | `MERGE` — idempotent, updates existing nodes | Incremental updates, re-runnable pipelines |

MERGE updates properties on existing nodes and only creates new ones when they don't exist yet.

In [5]:
# Re-merge with one updated and one new node
people2 = NodeSet(labels=["Person"], merge_keys=["email"])
people2.add({"email": "alice@example.com", "name": "Alice", "age": 31})  # birthday!
people2.add({"email": "frank@example.com", "name": "Frank", "age": 45})  # new person

people2.merge(driver)

query("MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name")

{'name': 'Alice', 'age': 31}
{'name': 'Bob', 'age': 25}
{'name': 'Carol', 'age': 35}
{'name': 'Dave', 'age': 28}
{'name': 'Eve', 'age': 32}
{'name': 'Frank', 'age': 45}


[{'name': 'Alice', 'age': 31},
 {'name': 'Bob', 'age': 25},
 {'name': 'Carol', 'age': 35},
 {'name': 'Dave', 'age': 28},
 {'name': 'Eve', 'age': 32},
 {'name': 'Frank', 'age': 45}]

### Multiple labels

Nodes can carry multiple labels. graphio generates the appropriate Cypher for any combination.

In [6]:
admins = NodeSet(labels=["Person", "Admin"], merge_keys=["email"])
admins.add({"email": "alice@example.com", "name": "Alice", "age": 31})
admins.merge(driver)

query("MATCH (p:Person:Admin) RETURN p.name AS name, labels(p) AS labels")

{'name': 'Alice', 'labels': ['Person', 'Admin']}


[{'name': 'Alice', 'labels': ['Person', 'Admin']}]

### Bulk load performance

The default batch size is **10,000 nodes per transaction** (tunable via `batch_size=`). graphio uses `UNWIND` batching under the hood — the Neo4j-recommended pattern:

```cypher
UNWIND $props AS properties
MERGE (n:Person {email: properties.email})
ON CREATE SET n = properties
ON MATCH SET n += properties
```

In [7]:
import random, time

N = 50_000
users = NodeSet(labels=["User"], merge_keys=["email"])
for i in range(N):
    users.add({"email": f"user{i}@example.com", "name": f"User {i}", "score": random.randint(1, 100)})

users.create_index(driver)

t0 = time.time()
users.merge(driver)
elapsed = time.time() - t0

print(f"Merged {N:,} nodes in {elapsed:.2f}s  ({N/elapsed:,.0f} nodes/sec)")

Merged 50,000 nodes in 0.79s  (63,012 nodes/sec)


In [8]:
query("MATCH (u:User) RETURN count(u) AS user_count")

{'user_count': 50000}


[{'user_count': 50000}]

---
## 2 — RelationshipSet: Bulk-loading relationships

A `RelationshipSet` holds relationships of the **same type** connecting nodes identified by their properties. You don't need internal Neo4j IDs — graphio matches nodes by whatever properties you specify.

The generated Cypher looks like this:

```cypher
UNWIND $rels AS rel
MATCH (a:Person), (b:Person)
WHERE a.email = rel.start_email
  AND b.email = rel.end_email
MERGE (a)-[r:KNOWS]->(b)
ON CREATE SET r = rel.properties
ON MATCH SET r += rel.properties
```

This means **node indexes are essential** — without them, every relationship load becomes a full scan.

In [9]:
# Helper to reset between sections
def clear_db():
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("Database cleared.")

clear_db()

people = NodeSet(labels=["Person"], merge_keys=["email"])
for person in [
    {"email": "alice@example.com", "name": "Alice"},
    {"email": "bob@example.com",   "name": "Bob"},
    {"email": "carol@example.com", "name": "Carol"},
    {"email": "dave@example.com",  "name": "Dave"},
    {"email": "eve@example.com",   "name": "Eve"},
]:
    people.add(person)

people.create_index(driver)
people.merge(driver)
print("Persons created.")

Database cleared.
Persons created.


In [10]:
knows = RelationshipSet(
    rel_type="KNOWS",
    start_node_labels=["Person"],
    end_node_labels=["Person"],
    start_node_properties=["email"],   # how to match the start node
    end_node_properties=["email"],     # how to match the end node
)

# .add(start_node_props, end_node_props, relationship_props)
knows.add({"email": "alice@example.com"}, {"email": "bob@example.com"},   {"since": 2018})
knows.add({"email": "alice@example.com"}, {"email": "carol@example.com"}, {"since": 2020})
knows.add({"email": "bob@example.com"},   {"email": "dave@example.com"},  {"since": 2021})
knows.add({"email": "carol@example.com"}, {"email": "eve@example.com"},   {"since": 2019})
knows.add({"email": "dave@example.com"},  {"email": "alice@example.com"}, {"since": 2022})

print(f"{knows}")
print(f"Relationships in memory: {len(knows.relationships)}")

<RelationshipSet (['Person']; ['email'])-[KNOWS]->(['Person']; ['email'])>
Relationships in memory: 5


In [11]:
# create_index on a RelationshipSet creates indexes for both start and end node match properties
knows.create_index(driver)
knows.merge(driver)
print("Done.")

Done.


In [12]:
query("""
MATCH (a:Person)-[r:KNOWS]->(b:Person)
RETURN a.name AS from, b.name AS to, r.since AS since
ORDER BY a.name
""")

{'from': 'Alice', 'to': 'Bob', 'since': 2018}
{'from': 'Alice', 'to': 'Carol', 'since': 2020}
{'from': 'Bob', 'to': 'Dave', 'since': 2021}
{'from': 'Carol', 'to': 'Eve', 'since': 2019}
{'from': 'Dave', 'to': 'Alice', 'since': 2022}


[{'from': 'Alice', 'to': 'Bob', 'since': 2018},
 {'from': 'Alice', 'to': 'Carol', 'since': 2020},
 {'from': 'Bob', 'to': 'Dave', 'since': 2021},
 {'from': 'Carol', 'to': 'Eve', 'since': 2019},
 {'from': 'Dave', 'to': 'Alice', 'since': 2022}]

### Relationships between different node types

In [13]:
companies = NodeSet(labels=["Company"], merge_keys=["name"])
companies.add({"name": "Acme Corp",    "industry": "Manufacturing"})
companies.add({"name": "GraphDB Inc",  "industry": "Technology"})
companies.add({"name": "Data Systems", "industry": "Technology"})
companies.create_index(driver)
companies.merge(driver)

works_at = RelationshipSet(
    rel_type="WORKS_AT",
    start_node_labels=["Person"],
    end_node_labels=["Company"],
    start_node_properties=["email"],
    end_node_properties=["name"],
)

works_at.add({"email": "alice@example.com"}, {"name": "GraphDB Inc"},  {"role": "Engineer",  "since": 2019})
works_at.add({"email": "bob@example.com"},   {"name": "Acme Corp"},    {"role": "Manager",   "since": 2020})
works_at.add({"email": "carol@example.com"}, {"name": "Data Systems"}, {"role": "Analyst",   "since": 2021})
works_at.add({"email": "dave@example.com"},  {"name": "GraphDB Inc"},  {"role": "Architect", "since": 2018})
works_at.add({"email": "eve@example.com"},   {"name": "Acme Corp"},    {"role": "Director",  "since": 2017})

works_at.create_index(driver)
works_at.merge(driver)
print("Done.")

Done.


In [14]:
query("""
MATCH (p:Person)-[r:WORKS_AT]->(c:Company)
RETURN p.name AS person, c.name AS company, r.role AS role
ORDER BY c.name, p.name
""")

{'person': 'Bob', 'company': 'Acme Corp', 'role': 'Manager'}
{'person': 'Eve', 'company': 'Acme Corp', 'role': 'Director'}
{'person': 'Carol', 'company': 'Data Systems', 'role': 'Analyst'}
{'person': 'Alice', 'company': 'GraphDB Inc', 'role': 'Engineer'}
{'person': 'Dave', 'company': 'GraphDB Inc', 'role': 'Architect'}


[{'person': 'Bob', 'company': 'Acme Corp', 'role': 'Manager'},
 {'person': 'Eve', 'company': 'Acme Corp', 'role': 'Director'},
 {'person': 'Carol', 'company': 'Data Systems', 'role': 'Analyst'},
 {'person': 'Alice', 'company': 'GraphDB Inc', 'role': 'Engineer'},
 {'person': 'Dave', 'company': 'GraphDB Inc', 'role': 'Architect'}]

---
## 3 — Advanced NodeSet features

### `preserve` — don't overwrite on MERGE

Some properties should only ever be written once, like `created_at`. Listing them in `preserve` means they are set on `ON CREATE` but skipped on `ON MATCH`.

In [15]:
clear_db()

users = NodeSet(labels=["Person"], merge_keys=["email"], preserve=["created_at"])
users.add({"email": "alice@example.com", "name": "Alice", "created_at": "2024-01-01"})
users.create_index(driver)
users.merge(driver)

# Merge again with a different created_at — it should NOT change
update = NodeSet(labels=["Person"], merge_keys=["email"], preserve=["created_at"])
update.add({"email": "alice@example.com", "name": "Alice Updated", "created_at": "2999-01-01"})
update.merge(driver)

query("MATCH (p:Person) RETURN p.name AS name, p.created_at AS created_at")
# name IS updated, created_at stays '2024-01-01'

Database cleared.
{'name': 'Alice Updated', 'created_at': '2024-01-01'}


[{'name': 'Alice Updated', 'created_at': '2024-01-01'}]

### `append_props` — accumulate values as a list

When a property is listed in `append_props`, each MERGE appends the new value to the existing list rather than overwriting it. Useful for event logs, tags, or any collected series.

In [16]:
sessions = NodeSet(labels=["User"], merge_keys=["user_id"], append_props=["logins"])
sessions.add({"user_id": "u1", "logins": "2024-01-10"})
sessions.create_index(driver)
sessions.merge(driver)

sessions2 = NodeSet(labels=["User"], merge_keys=["user_id"], append_props=["logins"])
sessions2.add({"user_id": "u1", "logins": "2024-01-15"})
sessions2.merge(driver)

query("MATCH (u:User {user_id: 'u1'}) RETURN u.logins AS login_history")
# → ['2024-01-10', '2024-01-15']

{'login_history': ['2024-01-10', '2024-01-15']}


[{'login_history': ['2024-01-10', '2024-01-15']}]

### `deduplicate` — drop duplicates in memory

When source data is messy, `deduplicate=True` silently skips nodes whose merge key already exists in the set — before anything hits Neo4j.

In [17]:
raw_data = [
    {"email": "alice@example.com", "name": "Alice"},
    {"email": "bob@example.com",   "name": "Bob"},
    {"email": "alice@example.com", "name": "Alice (duplicate)"},  # same email!
    {"email": "bob@example.com",   "name": "Bob (duplicate)"},    # same email!
]

clean = NodeSet(labels=["Person"], merge_keys=["email"], deduplicate=True)
for row in raw_data:
    clean.add(row)

print(f"Raw rows: {len(raw_data)}, after deduplication: {len(clean.nodes)}")

Raw rows: 4, after deduplication: 2


---
## 4 — OGM: Object Graph Mapper

For applications that need a proper data model, graphio ships a **Pydantic-based OGM**.

- Models are defined as Pydantic classes with `_labels` and `_merge_keys`
- Relationships are declared as class attributes using `Relationship(source, type, target)`
- The driver is set once on `Base` and shared by all models
- The OGM and the bulk datasets are **interoperable** — `Model.nodeset()` bridges both worlds

### 4.1 — Model definition & writing data

In [18]:
clear_db()

class Person(NodeModel):
    name: str
    age: int
    city: str
    email: str

    _labels     = ["Person"]
    _merge_keys = ["email"]

    knows: Relationship = Relationship("Person", "KNOWS", "Person")


Base.set_driver(driver)   # set once, shared by all models
Person.create_index()
print("Model ready.")

Database cleared.
Model ready.


In [19]:
# Create instances
alice = Person(name="Alice", age=31, city="Berlin",  email="alice@example.com")
bob   = Person(name="Bob",   age=25, city="Hamburg", email="bob@example.com")
carol = Person(name="Carol", age=35, city="Berlin",  email="carol@example.com")

# Merge nodes first, then attach relationships
for person in [alice, bob, carol]:
    person.merge_node()

alice.knows.add(bob)
alice.knows.add(carol)
bob.knows.add(carol)

for person in [alice, bob, carol]:
    person.merge_relationships()

print("Done.")

Done.


> **Note:** We use `merge_node()` + `merge_relationships()` separately here.
> The convenience method `merge()` recursively merges target nodes too, which causes infinite recursion
> when relationships form cycles (e.g. Alice→Bob→Alice). Splitting the steps avoids this.

### 4.2 — Querying nodes

Model fields become **query operators** at the class level. The result is always a list of model instances — fully typed, ready to use.

In [20]:
# Get all nodes
everyone = Person.match().all()
print(f"All persons ({len(everyone)}):")
for p in everyone:
    print(f"  {p.name}, age {p.age}, {p.city}")

All persons (3):
  Alice, age 31, Berlin
  Bob, age 25, Hamburg
  Carol, age 35, Berlin


In [21]:
# Equality filter
berliner = Person.match(Person.city == "Berlin").all()
print("People in Berlin:", [p.name for p in berliner])

People in Berlin: ['Alice', 'Carol']


In [22]:
# Comparison operators:  ==  !=  >  >=  <  <=
over_30 = Person.match(Person.age >= 30).all()
print("Age >= 30:", [p.name for p in over_30])

young = Person.match(Person.age < 30).all()
print("Age < 30: ", [p.name for p in young])

Age >= 30: ['Alice', 'Carol']
Age < 30:  ['Bob']


In [23]:
# String operators: .contains()  .starts_with()  .ends_with()
has_a = Person.match(Person.name.contains("a")).all()   # case-sensitive
print(".contains('a'):",   [p.name for p in has_a])

starts_a = Person.match(Person.name.starts_with("A")).all()
print(".starts_with('A'):", [p.name for p in starts_a])

ends_e = Person.match(Person.name.ends_with("e")).all()
print(".ends_with('e'):",   [p.name for p in ends_e])

.contains('a'): ['Carol']
.starts_with('A'): ['Alice']
.ends_with('e'): ['Alice']


In [24]:
# Combine multiple conditions — all must be true (AND)
berlin_seniors = Person.match(
    Person.city == "Berlin",
    Person.age >= 30
).all()
print("Berlin, age >= 30:", [p.name for p in berlin_seniors])

Berlin, age >= 30: ['Alice', 'Carol']


In [25]:
# .first() returns a single instance (or None)
alice = Person.match(Person.email == "alice@example.com").first()
print(f"Found: {alice.name}, age {alice.age}, {alice.city}")

Found: Alice, age 31, Berlin


### 4.3 — Traversing relationships

Relationship attributes work both on **instances** ("who does Alice know?") and at the **class level** ("find people who know someone in Hamburg").

In [26]:
# Instance traversal: follow relationships from a specific node
alice = Person.match(Person.name == "Alice").first()

alices_connections = alice.knows.match().all()
print(f"Alice knows: {[p.name for p in alices_connections]}")

Alice knows: ['Bob', 'Carol']


In [27]:
# Filter on the target node: Alice knows who, and that person is in Hamburg?
hamburg_connections = alice.knows.match(Person.city == "Hamburg").all()
print(f"Alice knows people in Hamburg: {[p.name for p in hamburg_connections]}")

Alice knows people in Hamburg: ['Bob']


In [28]:
# Class-level traversal with source node filter:
# Find everyone known by a person in Berlin
known_by_berliner = Person.match(Person.city == "Berlin").knows.all()
print("Known by Berlin residents:", [p.name for p in known_by_berliner])

Known by Berlin residents: ['Bob', 'Carol']


In [29]:
# Chain source filter + target filter:
# Find people in Hamburg who are known by someone in Berlin
result = Person.match(Person.city == "Berlin").knows.match(Person.city == "Hamburg").all()
print("Hamburg people known by Berlin people:", [p.name for p in result])

Hamburg people known by Berlin people: ['Bob']


### 4.4 — Filtering on relationship properties

Use `.filter()` (instead of `.match()`) on the relationship to add conditions on the **relationship's own properties** rather than the target node.

In [30]:
# First, re-create with `since` on the KNOWS relationships
clear_db()

class Person(NodeModel):
    name: str
    age: int
    city: str
    email: str

    _labels     = ["Person"]
    _merge_keys = ["email"]

    knows: Relationship = Relationship("Person", "KNOWS", "Person")

Base.set_driver(driver)
Person.create_index()

alice = Person(name="Alice", age=31, city="Berlin",  email="alice@example.com")
bob   = Person(name="Bob",   age=25, city="Hamburg", email="bob@example.com")
carol = Person(name="Carol", age=35, city="Berlin",  email="carol@example.com")
dave  = Person(name="Dave",  age=28, city="Munich",  email="dave@example.com")
eve   = Person(name="Eve",   age=32, city="Hamburg", email="eve@example.com")

alice.knows.add(bob,   properties={"since": 2018})
alice.knows.add(carol, properties={"since": 2022})
carol.knows.add(eve,   properties={"since": 2019})
dave.knows.add(alice,  properties={"since": 2021})

for person in [alice, bob, carol, dave, eve]:
    person.merge()

print("Done.")

Database cleared.
Done.


In [31]:
# .filter() on the relationship half: filter by relationship property
# Who does Alice know, where the relationship started after 2020?
alice = Person.match(Person.name == "Alice").first()

recent = alice.knows.filter(RelField("since") > 2020).all()
print("Alice knows (since > 2020):", [p.name for p in recent])

Alice knows (since > 2020): ['Carol']


In [32]:
# Combine: relationship property filter + target node filter
result = alice.knows.filter(RelField("since") >= 2018).match(Person.city == "Hamburg").all()
print("Alice knows (since >= 2018, in Hamburg):", [p.name for p in result])

Alice knows (since >= 2018, in Hamburg): ['Bob']


### 4.5 — Raw Cypher queries

When the built-in filter syntax isn't enough, `CypherQuery` lets you write arbitrary Cypher and still get back typed model instances. The query must use `n` as the return variable.

In [33]:
# Parameterised Cypher — still returns Person instances
result = Person.match(CypherQuery(
    "MATCH (n:Person) WHERE n.age > $min_age RETURN n",
    params={"min_age": 30}
)).all()

print("age > 30 via CypherQuery:")
for p in result:
    print(f"  {p.name}, age {p.age}")

age > 30 via CypherQuery:
  Alice, age 31
  Carol, age 35
  Eve, age 32


In [34]:
# Multi-hop traversal — something the built-in query builder doesn't handle yet
# "Who are the friends-of-friends of Dave?"
result = Person.match(CypherQuery("""
    MATCH (:Person {name: 'Dave'})-[:KNOWS]->()-[:KNOWS]->(n:Person)
    RETURN DISTINCT n
""")).all()

print("Friends-of-friends of Dave:", [p.name for p in result])

Friends-of-friends of Dave: ['Carol', 'Bob']


### 4.6 — `nodeset()` bridge: typed model + bulk speed

Calling `Model.nodeset()` returns a `NodeSet` pre-configured with the model's labels and merge keys. This lets you use the OGM for your data model definition while still loading data in bulk.

In [35]:
# Use the model as the single source of truth for labels/merge-keys
# but load data through the fast bulk path
bulk = Person.nodeset()
print(bulk)   # same labels and merge_keys as the model

bulk.add({"name": "Grace",  "age": 29, "city": "Munich",  "email": "grace@example.com"})
bulk.add({"name": "Heidi",  "age": 38, "city": "Berlin",  "email": "heidi@example.com"})
bulk.add({"name": "Ivan",   "age": 41, "city": "Hamburg", "email": "ivan@example.com"})
bulk.merge(driver)

# Back to OGM queries immediately
munich = Person.match(Person.city == "Munich").all()
print("Munich:", [p.name for p in munich])

<NodeSet (['Person']; ['email'])>
Munich: ['Dave', 'Grace']


---
## 5 — Putting it all together: the hybrid approach

Real pipelines rarely fit neatly into "only OGM" or "only bulk". The sweet spot is:

- **Define models once** — single source of truth for labels, merge keys, and relationship types
- **OGM writes** for small, authoritative data (a handful of companies, config nodes)
- **`Model.nodeset()`** and **`Relationship.dataset()`** for the heavy loads — same model config, bulk speed
- **OGM queries** to explore and verify the result

`Relationship.dataset()` on a class-level relationship attribute returns a `RelationshipSet` pre-configured from both endpoint models — no manual label/key wiring needed.

In [36]:
from graphio import NodeSet, RelationshipSet, NodeModel, Relationship, Base, CypherQuery, RelField

clear_db()

# ── Model definitions — single source of truth ─────────────────────────
class Company(NodeModel):
    name: str
    country: str

    _labels     = ["Company"]
    _merge_keys = ["name"]

    maintains: Relationship = Relationship("Company", "MAINTAINS",   "Project")
    employs:   Relationship = Relationship("Company", "EMPLOYS",     "Person")


class Project(NodeModel):
    name: str
    language: str
    stars: int

    _labels     = ["Project"]
    _merge_keys = ["name"]


class Person(NodeModel):
    name: str
    email: str
    role: str

    _labels     = ["Person"]
    _merge_keys = ["email"]

    contributes_to: Relationship = Relationship("Person", "CONTRIBUTES_TO", "Project")


Base.set_driver(driver)
for model in [Company, Project, Person]:
    model.create_index()

print("Models ready.")

Database cleared.
Models ready.


In [37]:
# ── Step 1: OGM writes for small, authoritative data ───────────────────
# Companies are few and curated — write them individually with full control.

graphdb = Company(name="GraphDB Inc",  country="Germany")
datasys = Company(name="Data Systems", country="UK")

graphdb.maintains.add(Project(name="graphio",   language="Python", stars=420))
graphdb.maintains.add(Project(name="cypherkit", language="Python", stars=155))
datasys.maintains.add(Project(name="neo4j-etl", language="Java",   stars=280))

graphdb.merge()
datasys.merge()

# Verify with OGM queries
for c in Company.match().all():
    projects = c.maintains.match().all()
    print(f"  {c.name}: {[p.name for p in projects]}")

  GraphDB Inc: ['graphio', 'cypherkit']
  Data Systems: ['neo4j-etl']


In [38]:
# ── Step 2: Bulk load persons via nodeset() ────────────────────────────
# Many records from an external source — use the fast bulk path,
# but the NodeSet is configured directly from the Person model.

person_data = [
    {"email": "alice@example.com", "name": "Alice", "role": "Backend Engineer"},
    {"email": "bob@example.com",   "name": "Bob",   "role": "DevOps"},
    {"email": "carol@example.com", "name": "Carol", "role": "Data Engineer"},
    {"email": "dave@example.com",  "name": "Dave",  "role": "Graph Architect"},
    {"email": "eve@example.com",   "name": "Eve",   "role": "Frontend Engineer"},
]

persons_set = Person.nodeset()   # labels=["Person"], merge_keys=["email"] — from the model
for row in person_data:
    persons_set.add(row)
persons_set.merge(driver)

print(f"Persons in graph: {len(Person.match().all())}")

Persons in graph: 5


In [39]:
# ── Step 3: Bulk load relationships via Relationship.dataset() ─────────
# dataset() on a relationship attribute returns a RelationshipSet
# fully configured from the source and target model classes.

contributes_set = Person.contributes_to.dataset()  # Person -[CONTRIBUTES_TO]-> Project
employs_set     = Company.employs.dataset()         # Company -[EMPLOYS]-> Person

contributes_set.add({"email": "alice@example.com"}, {"name": "graphio"},   {"commits": 312})
contributes_set.add({"email": "dave@example.com"},  {"name": "graphio"},   {"commits": 198})
contributes_set.add({"email": "carol@example.com"}, {"name": "neo4j-etl"}, {"commits": 87})
contributes_set.add({"email": "bob@example.com"},   {"name": "cypherkit"}, {"commits": 54})
contributes_set.add({"email": "eve@example.com"},   {"name": "graphio"},   {"commits": 23})

employs_set.add({"name": "GraphDB Inc"},  {"email": "alice@example.com"})
employs_set.add({"name": "GraphDB Inc"},  {"email": "dave@example.com"})
employs_set.add({"name": "Data Systems"}, {"email": "carol@example.com"})
employs_set.add({"name": "Data Systems"}, {"email": "bob@example.com"})
employs_set.add({"name": "GraphDB Inc"},  {"email": "eve@example.com"})

for rs in [contributes_set, employs_set]:
    rs.create_index(driver)
    rs.merge(driver)

print("Relationships loaded.")

Relationships loaded.


In [40]:
# ── Step 4: OGM queries to explore the result ──────────────────────────

# Which projects does GraphDB Inc maintain?
graphdb = Company.match(Company.name == "GraphDB Inc").first()
print("GraphDB maintains:", [p.name for p in graphdb.maintains.match().all()])

GraphDB maintains: ['graphio', 'cypherkit']


In [41]:
# Top contributors to GraphDB projects — CypherQuery for aggregation
top = Person.match(CypherQuery("""
    MATCH (n:Person)-[r:CONTRIBUTES_TO]->(proj:Project)<-[:MAINTAINS]-(:Company {name: 'GraphDB Inc'})
    WITH n, sum(r.commits) AS total ORDER BY total DESC
    RETURN n
""")).all()

print("Top contributors to GraphDB Inc projects:")
for p in top:
    print(f"  {p.name} ({p.role})")

Top contributors to GraphDB Inc projects:
  Alice (Backend Engineer)
  Dave (Graph Architect)
  Bob (DevOps)
  Eve (Frontend Engineer)


In [42]:
# External contributors: contribute to a project but aren't employed by its company
externals = Person.match(CypherQuery("""
    MATCH (n:Person)-[:CONTRIBUTES_TO]->(proj:Project)<-[:MAINTAINS]-(c:Company)
    WHERE NOT (c)-[:EMPLOYS]->(n)
    RETURN DISTINCT n
""")).all()

print("External contributors:", [p.name for p in externals])

External contributors: ['Bob']


---
## Cleanup

In [43]:
# Uncomment to wipe the database
# clear_db()

driver.close()
print("Driver closed.")

Driver closed.
